In [1]:
# ==========================================
# 🔑 FRED API 키 (입력 필수!)
# ==========================================
FRED_API_KEY = "9c3f2227440e6d8c815f7996a4d253b5"

In [2]:
# ==========================================
# 🔑 FRED API 키 (입력 필수!)
# ==========================================
FRED_API_KEY = "9c3f2227440e6d8c815f7996a4d253b5"

import yfinance as yf
import pandas as pd
import numpy as np
import requests
from sklearn.preprocessing import StandardScaler
import warnings
warnings.filterwarnings('ignore')

print("🚀 [Data Factory V3] 고성능 퀀트 센서 26종 통합 수집 및 엔지니어링 시작...")

# 1. 수집 기간 설정 (예열 기간 고려하여 2009년부터 수집)
download_start = '2009-01-01'
actual_start_date = '2010-01-01'
end_date = '2026-04-02'

# ==========================================
# 1. 자산 가격 및 시장 폭(Breadth) 데이터 수집 (Yahoo)
# ==========================================
assets = ['SPY', 'QQQ', 'EEM', 'TLT', 'IEF', 'LQD', 'SHV', 'GLD', 'DBC', 'VNQ', 'RSP']
df_raw = yf.download(assets, start=download_start, end=end_date)
df_assets = df_raw['Adj Close'] if 'Adj Close' in df_raw.columns else df_raw['Close']
df_assets.dropna(inplace=True)

# ==========================================
# 2. 거시 경제 및 원자재 수집 (Yahoo)
# ==========================================
macro_tickers = {
    '^VIX': 'VIX', 
    'DX-Y.NYB': 'DXY',  
    'KRW=X': 'USDKRW',
    'CL=F': 'WTI_Oil',
    'HG=F': 'Copper',
    'GC=F': 'Gold'
}
df_macro_yf_raw = yf.download(list(macro_tickers.keys()), start=download_start, end=end_date)
df_macro_yf = df_macro_yf_raw['Adj Close'] if 'Adj Close' in df_macro_yf_raw.columns else df_macro_yf_raw['Close']
df_macro_yf.rename(columns=macro_tickers, inplace=True)

# ==========================================
# 3. FRED API 데이터 수집 (금리, 스프레드 + 🌟 유동성/고용 3대장 추가!)
# ==========================================
def get_fred_api_data(series_id, api_key):
    url = f"https://api.stlouisfed.org/fred/series/observations?series_id={series_id}&api_key={api_key}&file_type=json"
    response = requests.get(url)
    if response.status_code != 200: 
        print(f"⚠️ FRED API Error: {series_id}")
        return pd.Series()
    data = response.json()['observations']
    df = pd.DataFrame(data)
    df['date'] = pd.to_datetime(df['date'])
    df['value'] = pd.to_numeric(df['value'], errors='coerce')
    df.set_index('date', inplace=True)
    return df['value']

# 💡 친구분 추천 지표 3개 추가: WM2NS(M2), WALCL(연준 대차대조표), ICSA(실업수당)
fred_series = {
    'DGS10': 'US10Y', 
    'DGS2': 'US2Y', 
    'DGS3MO': 'US3M', 
    'BAMLH0A0HYM2': 'HY_Spread',
    'WM2NS': 'M2_Supply',        # 신규!
    'WALCL': 'Fed_Balance',      # 신규!
    'ICSA': 'Jobless_Claims'     # 신규!
}
fred_list = []
for series_id, col_name in fred_series.items():
    s = get_fred_api_data(series_id, FRED_API_KEY)
    s.name = col_name
    fred_list.append(s)

# 주간/월간 데이터이므로 빈 날짜가 많음 -> ffill()로 채워줌
df_macro_fred = pd.concat(fred_list, axis=1).loc[download_start:end_date].ffill()

# ==========================================
# 4. 병합 전 인덱스 타임존 정리 및 1차 병합
# ==========================================
df_assets.index = pd.to_datetime(df_assets.index).tz_localize(None).normalize()
df_macro_yf.index = pd.to_datetime(df_macro_yf.index).tz_localize(None).normalize()
df_macro_fred.index = pd.to_datetime(df_macro_fred.index).tz_localize(None).normalize()

df_raw_combined = pd.concat([df_assets, df_macro_yf, df_macro_fred], axis=1, join='inner').ffill()

# ==========================================
# 5. 💡 26개 퀀트 파생 지표 일괄 계산
# ==========================================
df = df_raw_combined.copy()

# [A~E 기존 파생 지표 유지]
df['VIX_ret'] = df['VIX'].pct_change()
df['VIX_MA5_Diff'] = df['VIX'] - df['VIX'].rolling(window=5).mean()
df['Volatility_20d'] = df['SPY'].pct_change().rolling(window=20).std() * np.sqrt(252)
df['Vol_Ratio'] = df['Volatility_20d'] / (df['SPY'].pct_change().rolling(window=60).std() * np.sqrt(252))
df['DXY_ret'] = df['DXY'].pct_change()
df['USDKRW_ret'] = df['USDKRW'].pct_change()
df['US10Y_diff'] = df['US10Y'].diff()
df['spread_10y2y'] = df['US10Y'] - df['US2Y']
df['spread_10y3m'] = df['US10Y'] - df['US3M']
df['HY_spread_ret'] = df['HY_Spread'].diff() 
df['GOLD_ret'] = df['Gold'].pct_change()
df['OIL_ret'] = df['WTI_Oil'].pct_change()
df['Copper_Gold_Ratio'] = df['Copper'] / df['Gold']
df['Market_Breadth'] = df['RSP'] / df['SPY']
df['Equity_vs_Bond'] = df['SPY'] / df['EEM']
df['SPY_ret'] = df['SPY'].pct_change()
df['SPY_Log_Ret'] = np.log(df['SPY'] / df['SPY'].shift(1))
df['SPY_MA20_Diff'] = (df['SPY'] / df['SPY'].rolling(window=20).mean()) - 1
df['MA200_Dist'] = df['SPY'] / df['SPY'].rolling(window=200).mean()

delta = df['SPY'].diff()
gain = (delta.where(delta > 0, 0)).rolling(window=14).mean()
loss = (-delta.where(delta < 0, 0)).rolling(window=14).mean()
rs = gain / loss
df['RSI'] = 100 - (100 / (1 + rs))

exp1 = df['SPY'].ewm(span=12, adjust=False).mean()
exp2 = df['SPY'].ewm(span=26, adjust=False).mean()
df['MACD'] = exp1 - exp2
df['Month'] = df.index.month

# 🌟 [F. 신규 유동성 & 고용 지표 계산] 🌟
# 약 3개월(60일)간의 유동성 변화율 추적
df['M2_Growth'] = df['M2_Supply'].pct_change(60)
df['Fed_BS_Growth'] = df['Fed_Balance'].pct_change(60)
# 실업수당 청구 건수 노이즈 제거 (20일 이평선)
df['Jobless_Claims_MA'] = df['Jobless_Claims'].rolling(window=20).mean()

# 윈도우 계산으로 인해 발생한 초기 결측치 제거 후, 원하는 날짜부터 자르기
df.dropna(inplace=True)
df = df.loc[actual_start_date:]

# ==========================================
# 6. 타겟 피처 스케일링 (신규 지표 3개 포함 총 25개)
# ==========================================
features_all = [
    "VIX_ret", "VIX", "VIX_MA5_Diff", "Vol_Ratio", "Volatility_20d", # Volatility_20d 추가됨
    "DXY_ret", "USDKRW_ret", "DXY",
    "US10Y_diff", "US10Y", "spread_10y2y", "spread_10y3m",
    "GOLD_ret", "OIL_ret",
    "HY_spread_ret", "SPY_ret", "SPY_Log_Ret", "SPY_MA20_Diff", "Equity_vs_Bond",
    "RSI", "MACD", "Month", 
    "Copper_Gold_Ratio", "Market_Breadth", "MA200_Dist",
    "M2_Growth", "Fed_BS_Growth", "Jobless_Claims_MA" # 🌟 신규 지표 스케일링 대상 추가!
]

scaler = StandardScaler()
df_final_scaled = df.copy()
df_final_scaled[features_all] = scaler.fit_transform(df[features_all])

print(f"\n📊 26개 피처 정제 및 스케일링 완료! 최종 데이터 형태: {df_final_scaled.shape}")

🚀 [Data Factory V3] 고성능 퀀트 센서 26종 통합 수집 및 엔지니어링 시작...
YF.download() has changed argument auto_adjust default to True


[*********************100%***********************]  11 of 11 completed
[*********************100%***********************]  6 of 6 completed



📊 26개 피처 정제 및 스케일링 완료! 최종 데이터 형태: (4086, 49)
